<a href="https://colab.research.google.com/github/chrismji/Datasci-and-Ai-in-Bus/blob/main/HW3_PatientNoShows.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# HW3: Predicting Patient No-Shows

### Background

You've been hired as the manager of SHMC, a medical center in Brazil. As part of its business model, SHMC usually employs doctors on demand, meaning that doctors are asked to come only when there are appointments. However, once a doctor has been asked to come, you must pay him or her regardless of whether the patient shows up or not. You want to use data on previous appointments to figure out whether you can predict which patients are going to show up, so that you can plan better how many doctors to call.

[The data can be downloaded at this link](https://drive.google.com/uc?download=export&id=1iM0CSzfNckc565zwpb4VpHjEVLgckLPE) in a file called SHMC_NoShows.csv.


Here are some libraries you will need

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn import metrics
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_score, recall_score, f1_score


### Loading the data

**1. Load the data referenced by the path above into a pandas data frame. Report on the total number of rows and columns, and show the first few rows in the dataset.**

Here's a brief summary of the features in the data:

1. PatientId - Identification of a patient.
2. AppointmentID - Identification of each appointment.
3. Gender - Male or female.
4. ScheduledDay - The day someone called or registered the appointment.
6. AppointmentDay - The day of the actual appointment, when the patient has to visit the doctor.
7. Age = How old is the patient.
8. Neighborhood - Place where the patient lives.
9. Scholarship - Whether the patient receives financial help.(0=No, 1=Yes)
10. Hypertension - Whether the patient suffers from hypertension.(0=No, 1=Yes)
11. Diabetes - Whether the patient suffers from diabetes.(0=No, 1=Yes)
12. Alcoholism - Whether the patient suffers from alcoholism.(0=No, 1=Yes)
13. Handicap - Whether the patient is handicapped (0=No, 1=Yes)
14. SMS_received - Whether 1 or more SMS messages were sent to the patient (0=No, 1=Yes)
15. No-show - Whether the patient missed the appointment.

### Exploratory Data Analysis

**2. Make sure Python recognizes `ScheduledDay` and `AppointmentDay` as timestamps by using `.to_datetime()`. What is the earliest and latest date of  AppointmentDay, and calculate the number of days between these two?**

**3. If you take a look at both dates, you will see that `ScheduledDay` includes a specific time with the date, but `AppointmentDay` doesn't. In order to compare both fields more easily, apply `.normalize()` to `ScheduledDay` (i.e., keep only the dates).**

**4. Create a new feature called 'TimeInAdvance' which is the difference in days between `ScheduledDay` and `AppointmentDay`.  (you may need to apply `dt.days`  to a datetime object to make sure this is a numeric). What is the longest advance time that someone scheduled an appointment? What do you expect the relationship between `TimeInAdvance` and the probability of a No-Show to be?**

**5. Explore the data set using EDA functions we have discussed in class.  Look at each feature.  Do any of them contain missing or weird values or outliers that concern you, given the feature descriptions above?  Comment on any unusual values you see in any of the columns.  Remove observations that contain any strange observations you have no explanation for, and report on how many rows were removed.**

**6. Create a new column from `Gender` called `IsFemale` that is 1 if the patient is female and 0 otherwise. Do the same for the No-show variable. It should be 1 if the person didn't show and 0 otherwise. This is your target variable. Drop the original columns for these two.**

**7. Now we explore the `Neighborhood` feature.  It is a categorical with many levels (as can be seen from `value_counts`).  To simplify the analysis we are going to create a new verison of `Neighborhood` by re-labelling any neighborhoods that appears LESS THAN 3000 times and give them the label Neighborhood="OTHER".  Show a table of value_counts of your *new* Neighborhood feature. Create dummy variables for your new Neighborhood feature, add the dummies to your data frame, and then drop the original Neighborhood column from the data frame.**

**8. There are many patients that have historically made more than one appointment. This data set contains multiple observations per person, so we can use their past appointments as features in predicting whether the current appointment will be a no-show.**

**For example, the fact that a patient has failed to show up in the past may be predictive of the patient not showing up in the future. We want to create a variable that calculates the number of no_shows BEFORE the current appointment.**

**Create a new column called `PreviousNoShows` that for each appointment includes the number of *previous* appointments to which the patient did not show. For example, if the `PatientID` did not show to their first appointment, `PreviousNoShows` should be 0 for the first appointment and 1 for the second one (if there is any). (We have provided code for this part below, but take a look at it to see if it makes sense to you, and edit the code to use your data frame name).**

**Now, create a line plot showing the relationship between PreviousNoShows and No-Show for the current appointment.   For each value of PreviousNoShows, plot the percent of No-Shows for the current appointment (Hint: use `groupby()`).  Does there seem to be a relationship? That is, does the number of PREVIOUS no-shows seem to be predictive of a no-show for the CURRENT appointment?  Comment on what you see.**

In [ ]:
## sample code for calculating PreviousNoShows- make sure to replace "df" with your data frame name

# df = df.sort_values(['PatientId', 'ScheduledDay'])
# df['PreviousNoShows'] = df.groupby(['PatientId'])['No-show'].cumsum()- df['No-show']


### Modelling with Decision Trees

**9. Build a model to predict No-Show using a DecisionTreeClassifier:**
* **First, drop the columns we won't be needing: `PatientId`, `AppointmentID`,
`ScheduledDay`, and `AppointmentDay`.**
* **Split the data into training and test (80/20) - using `random_state=99`.**  
* **Apply 5-fold `GridSearchCV()` *only to the training data* (`X_train` and `y_train`)  to find the optimal values of `max_depth` (search values between 1 and 51 in steps of 10) and `min_samples_split` (search values from 5 to 25 in steps of 5). Use F-measure (aka f1) as the scoring metric.  Report the optimal parameter values AND the optimal F1 score.**


**10: Assign the optimal model that comes out of `GridSearchCV` to `final_model`.  Plot only the first three levels of the final_model tree, by using `.plot_tree()` with the option `max_depth=3` (see code below).  What are the rules to get to the leaf node that has the HIGHEST probability of NO-SHOW in these first three levels? Comment on these rules, do they make sense to you? Are they intuitive?**

In [ ]:
final_model = #####
plt.figure(figsize=(25,10))
plot_tree(final_model, feature_names=X_train.columns, class_names=['Show', 'No-Show'], filled=True, max_depth=3);
plt.show()

**11. Calculate the feature importance values for the features in `final_model`.  Comment on each of the top 5 features and discuss how each one impacts no-show probability.  Comment on how you might you deal with this knowledge in making your office more efficient.**

**12. Use your `final_model` to predict values for `X_test` to get `y_pred`. Using `y_pred` and `y_test`, calculate precision and recall using the functions `precision_score` and `recall_score`**

## Exploring Different Thresholds

**13. The precision/reall values above were calculated using a threshold of 0.5 (this is the default in the tree fitting function).  As we discussed in class, changing the threshold will have an effect on precision and recall.  Plot boxplots of the predicted probabilities for no-shows and non no-shows in the test set.  Pick a threshold that you think will separate the two best.**


**14. Calculate precision and recall for your new threshold and compare it to the precision/recall for the default threshold of 0.5.  Comment on the tradeoff between the two thresholds and how it changes false positives and negatives.  Which threshold would you choose and why?**